# Language Detection Model

This project focuses on Language Detection, which is a multi-class classification task in the field of NLP. Our goal is to train a model to automatically identify/predict the language of a given text.

While identifying a language might seems obvious to us human beings, it is a complex challenge for a computer. The difficulty lies in the fact that many languages share the exact same alphabet. For example, English, French, and Spanish all use Latin letters, while several Middle Eastern and Asian languages might use similar-looking characters. A machine cannot "read" like we do. instead, it must uncover hidden patterns and statistical fingerprints, such as the frequency of specific character combinations (n-grams) or unique word structures that define each language.

We are using a [Kaggle dataset that contains 10,000 sentences across 17 different languages](https://www.kaggle.com/datasets/basilb2s/language-detection), including English, Arabic, Spanish, and others. To evaluate our success, we will use the Macro-Average F1 score, which ensures that the model is accurate across all languages, regardless of how many sentences they have in the dataset.

## Project Members

| Name | Last 4 Digits of ID |
| :--- | :--- |
| Almog S | 5890 |
| Eliad B | 6279 |

## Import Dependencies

In [1]:
import pandas as pd
import numpy as np
import sys
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import classification_report, accuracy_score

text_col = 'Text'
label_col = 'Language'

def load_dataset(file_path):
    return pd.read_csv(file_path, encoding='utf-8')

def view_features(raw, vectorized_data, row_idx):
    row = vectorized_data[row_idx]
    nonzero_indices = row.indices # if we wont filter the data, we'll see most of the columns are filled with 0.0
    tfidf_values = row.data
    
    all_feature_names = vectorizer.get_feature_names_out()
    nonzero_feature_names = [all_feature_names[i] for i in nonzero_indices]
    table = pd.DataFrame([tfidf_values], columns=nonzero_feature_names)
    display(table)

## Loading the dataset

Our dataset is stored in a CSV file that contains 2 columns: the **text**s and its corresponding **language** label. A DataFrame is the perfect tool for organizing the data and the labels into a clean, tabular structure that is easy to manipulate.

In [2]:
df = load_dataset('dataset.csv')
print(f"{df.shape[0]} sentences have been processed.")
df.head()

10337 sentences have been processed.


,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English


## Data Splitting

First, we split the data into train set and test set. We chose to split the data into 80% training and 20% testing.

In [3]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df[text_col], df[label_col], test_size=0.2)#, random_state=42)

train_preview = pd.concat([X_train_raw, y_train], axis=1)
train_preview.columns = [text_col, label_col]

test_preview = pd.concat([X_test_raw, y_test], axis=1)
test_preview.columns = [text_col, label_col]

print("### Table 1: Train Set Snapshot (First 5 Rows)")
display(train_preview.head())
print("### Table 2: Test Set Snapshot (First 5 Rows)")
display(test_preview.head())

### Table 1: Train Set Snapshot (First 5 Rows)


,Text,Language
5000,Con excepción de ciertas personas remuneradas ...,Spanish
8131,iletişimde kalın.,Turkish
5452,Mantente en contacto.,Spanish
6142,Вандализм в Википедии — явно вредительское доб...,Russian
4408,"Enkele voorbeelden zijn DBpedia, EverybodyWiki...",Dutch


### Table 2: Test Set Snapshot (First 5 Rows)


,Text,Language
8306,[6] Före utgången av 2001 fanns Wikipedia på 1...,Sweedish
7700,che ne dici di avere una pizza per cena stasera?,Italian
2231,"நீங்கள் ஒருவருக்கு நன்றி சொல்ல விரும்பினால், அ...",Tamil
1194,this is a no-brainer means it's a really easy ...,English
980,Trained models derived from biased data can re...,English


## Feature Engineering

In this step, we transform the raw text into a numerical format that the machine learning algorithm can understand. We use the **TF-IDF Vectorizer with Character N-grams** (ranging from 1 to 3 characters): Instead of looking at complete words, the model breaks down each sentence into small sequences of single letters, pairs, and triplets. This is very effective for language detection because every language has a unique statistical "fingerprint" based on how often specific character combinations appear. This method allows the model to differentiate between languages that share the same alphabet and stay accurate even when there are typos or unfamiliar words.

**How does TF-IDF work?**

The value in each cell is calculated using this formula:

$$Score = TF \times IDF$$

Where:

$$TF = \frac{\text{Number of times the sequence appears in the row}}{\text{Total number of sequences in that row}}$$

$$IDF = \log \left( \frac{\text{Total number of rows in the DataFrame}}{\text{Number of rows containing the sequence}} \right)$$

In other words, $TF$ indicates how many times a specific letter sequence appears in a single sentence, while $IDF$ indicates how rare this sequence is across the entire dataset.

**The higher the score, the more important that character sequence is for identifying the language.**

In [4]:
# analyzer='char' tells the model to look at characters, not words.
# ngram_range=(1, 3) looks at single letters, pairs, and triplets.
# norm='l2' will make our data normalized
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), norm='l2')
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(X_train.shape)
print(X_test.shape)

(8269, 205249)
(2068, 205249)


Below are examples of the non-zero features of 2 sentences.

In [5]:
for i in [0, 1, 2]:
    print(f"Train example {i}: '{X_train_raw.iloc[i]}'")
    view_features(X_train_raw, X_train, i)

for i in [0, 1, 2]:
    print(f"Test example {i}: '{X_test_raw.iloc[i]}'")
    view_features(X_test_raw, X_test, i)

Train example 0: 'Con excepción de ciertas personas remuneradas por la Fundación Wikimedia,[131]​ el resto, conocidos en la jerga de Wikipedia como wikipedistas, actúan siempre de manera gratuita y voluntaria.'


,co,on,n,e,ex,xc,ce,ep,pc,ci,...,y vo,vol,volu,olun,lunt,unta,ntar,tari,aria,ria.
0,0.058856,0.050745,0.074516,0.047751,0.030182,0.051506,0.022233,0.032772,0.06135,0.090292,...,0.076753,0.046194,0.051506,0.055646,0.057404,0.050601,0.046295,0.0532,0.044897,0.06135


Train example 1: 'iletişimde kalın.'


,de,e,im,de,il,le,et,ti,iş,şi,...,işim,şimd,imde,mde,de k,e ka,kal,kalı,alın,lın.
0,0.044944,0.037592,0.0719,0.059611,0.06453,0.05147,0.057362,0.051101,0.140315,0.147979,...,0.202408,0.17597,0.202408,0.191097,0.159289,0.146053,0.161825,0.194311,0.191097,0.224458


Train example 2: 'Mantente en contacto.'


...

,1,18,18,2,20,200,a,av,av,f,...,wiki,å,åk,åk.,ån,ång,ånge,ör,öre,öre
0,0.064212,0.098599,0.12317,0.055057,0.056306,0.063328,0.02794,0.066445,0.080658,0.074825,...,0.046058,0.068135,0.092178,0.13046,0.085977,0.097085,0.12317,0.06605,0.09745,0.121343


Test example 1: 'che ne dici di avere una pizza per cena stasera?'


,a,av,ave,c,ce,cen,d,di,di,dic,...,una,ve,ver,vere,za,za,za p,zz,zza,zza
0,0.032574,0.077466,0.097746,0.039176,0.07776,0.111198,0.064732,0.102097,0.078363,0.099044,...,0.081721,0.0432,0.056931,0.115423,0.07388,0.099238,0.148801,0.101295,0.106281,0.124502


Test example 2: 'நீங்கள் ஒருவருக்கு நன்றி சொல்ல விரும்பினால், அது மிகவும் வகையானது என்று நீங்கள் கூறலாம், அதைச் செய்ததற்கு மிக்க நன்றி, அது உங்களுக்கு மிகவும் வகையானது, யாராவது இருந்தால் அது உங்களுக்கு மிகவும் வகையானது.'


,அ,அத,அது,அதை,இ,இர,இரு,உ,உங,உங்,...,்பின,்ற,்றி,்றி,"்றி,",்று,்று,்ல,்ல,்ல வ
0,0.087352,0.103253,0.097118,0.029454,0.021573,0.024391,0.024746,0.045557,0.052619,0.052619,...,0.031565,0.061942,0.054633,0.030677,0.039498,0.023093,0.024799,0.022268,0.030883,0.034168


## Model Training (KNN)

Now that our text is represented as a matrix of numerical vectors (TF-IDF), we can apply the **KNN algorithm**. 

KNN predicts the language of a new sentence by looking at the $k$ most similar sentences in our training data.

The logic of our algorithm implementation is structured as follows:

- We use the **cosine similarity score** to check how similar a test sentence is to the sentences in our training set. The score ranges from [-1,1] and is calculated using the dot product and magnitude:

$$\displaystyle{\text{Cosine similarity} = \cos \left( \theta \right) = \frac{ A \cdot B }{ \left\| A \right\| \cdot \left\| B \right\| } }$$

In this formula, $A$ represents a row vector from $X_{\text{test}}$ and $B$ represents a column vector from $X_{\text{train}}^T$ - the transpose of $X_{\text{train}}$.

Since our data is already L2 normalized by the TF-IDF vectorizer, the magnitude of each vector is 1 ($\|A\| = 1$ and $\|B\| = 1$). This simplifies the formula:

$$\displaystyle{ \text{Cosine similarity} = \frac{ A \cdot B }{ 1 \cdot 1 } = A \cdot B }$$
This means we only need to calculate the **dot product** between the test row and all training rows (using $X_{\text{test}} \cdot X_{\text{train}}^T$). 

So the result of $X_{\text{test}} \cdot X_{\text{train}}^T$ is a matrix where each cell shows the cosine similarity score."

<img src="https://raw.githubusercontent.com/Almoger/Language-Detection-ML/refs/heads/main/Images/dot_product_example.png" alt="error" width="700" height="700">

- For each test sentence (each row), we **sort the similarity scores and pick the $k$ nearest neighbors** - these are the train sentences with the highest similarity scores. 
- We check the labels (languages) of these $k$ neighbors. The language that appears most often among them (based on the majority rule) is chosen as the final prediction for the sentence.

**Note**: We will explore different K-values using Cross-Validation at the [Evaluation](#Evaluation) step to find the optimal hyperparameters.

In [6]:
class KNN(BaseEstimator, ClassifierMixin):
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = np.array(y)

    def predict(self, X_test):
        similarities = X_test.dot(self.X_train.T).toarray() # dot product
        predictions = []
        
        for row in similarities:
            neighbors_indexes = np.argsort(row)[-self.k:] # get k indexes of the highest values (nearest neighbors)
            k_neighbor_labels = self.y_train[neighbors_indexes] # get their labels

            unique_labels, counts = np.unique(k_neighbor_labels, return_counts=True) # count how much times each label appears
            best_label_index = np.argmax(counts) # index of label that appears the most
            best_label = unique_labels[best_label_index] # retreive the label
            predictions.append(best_label)
            
        return np.array(predictions)

## Evaluation - what are the best parameters?

This step will help us answer two key questions:

- How accurate is the model on new text, one that it has never seen before?

- How often does it make mistakes?

We will use the Accuracy score and the Macro F1-score to see the full picture and ensure the model performs well across all 17 languages.

In [7]:
from sklearn.model_selection import GridSearchCV

best_knn = None
best_vectorizer = None
best_f1_macro_score = 0.0

ngram_options = [(1, 2), (1, 3), (2, 3), (2, 4)]
k_options = [5, 11, 53, 89]
all_results = []

print("Starting Experiments...")

for ngram in ngram_options:
    print(f"Testing Feature Engineering: ngram_range={ngram}")
    
    temp_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=ngram, norm='l2')
    X_train_vec = temp_vectorizer.fit_transform(X_train_raw)
    
    knn_grid = GridSearchCV(
        KNN(), 
        param_grid={'k': k_options}, 
        cv=5, 
        scoring='f1_macro'
    )
    
    knn_grid.fit(X_train_vec, y_train)

    if knn_grid.best_score_ > best_f1_macro_score:
        best_f1_macro_score = knn_grid.best_score_
        best_knn = knn_grid.best_estimator_
        best_vectorizer = temp_vectorizer
    
    cv_res = knn_grid.cv_results_
    for i in range(len(k_options)):
        all_results.append({
            'ngram_range': ngram,
            'k': cv_res['param_k'][i],
            'mean_f1_macro': cv_res['mean_test_score'][i],
        })

full_results_df = pd.DataFrame(all_results).sort_values(by='mean_f1_macro', ascending=False)
display(full_results_df)

best_row = full_results_df.iloc[0]
best_ngram = best_row['ngram_range']
best_k = best_row['k']
best_f1_macro_score = best_row['mean_f1_macro']

print(f"\n--- Best Configuration Found ---")
print(f"NGram Range: {best_ngram}")
print(f"K Value: {best_k}")
print(f"Validation F1-Macro: {best_f1_macro_score}")

Starting Experiments...
Testing Feature Engineering: ngram_range=(1, 2)
Testing Feature Engineering: ngram_range=(1, 3)
Testing Feature Engineering: ngram_range=(2, 3)
Testing Feature Engineering: ngram_range=(2, 4)


,ngram_range,k,mean_f1_macro
14,"(2, 4)",53,0.984367
9,"(2, 3)",11,0.984346
13,"(2, 4)",11,0.984232
10,"(2, 3)",53,0.982643
15,"(2, 4)",89,0.982272
5,"(1, 3)",11,0.982211
11,"(2, 3)",89,0.981926
12,"(2, 4)",5,0.977274
4,"(1, 3)",5,0.976823
8,"(2, 3)",5,0.976599



--- Best Configuration Found ---
NGram Range: (2, 4)
K Value: 53
Validation F1-Macro: 0.9843673970269503


## Model Testing

Now, we want to check if our model is actually successful. It is important to know if the model truly "learned" how to identify languages or if the results were just a matter of luck. To do this, we will test the model on the Test Set data that the model has never seen before.

In [8]:
X_train = best_vectorizer.fit_transform(X_train_raw)
X_test = best_vectorizer.transform(X_test_raw)
y_pred = best_knn.predict(X_test)
results_table = pd.DataFrame({'Text': X_test_raw[:20], 'Actual': y_test[:20], 'Predicted': y_pred[:20]})
display(results_table)

,Text,Actual,Predicted
8306,[6] Före utgången av 2001 fanns Wikipedia på 1...,Sweedish,Sweedish
7700,che ne dici di avere una pizza per cena stasera?,Italian,Italian
2231,"நீங்கள் ஒருவருக்கு நன்றி சொல்ல விரும்பினால், அ...",Tamil,Tamil
1194,this is a no-brainer means it's a really easy ...,English,English
980,Trained models derived from biased data can re...,English,English
2101,"[57] வழமையாக, கட்டுரையொன்றில் மேற்கொள்ளப்படும்...",Tamil,Tamil
2928,ele encontrou durante suas viagens ou em uma e...,Portugeese,Portugeese
3900,Les notions de percept et de concept comme phé...,French,French
3090,não muito.,Portugeese,Portugeese
6710,[19] Wikipedia er blevet kritiseret for at udv...,Danish,Danish


Great! our predictions match the results.

## DIY

Enter your desired text and check our model!

In [9]:
input_text = 'Chi va piano, va sano e va lontano' # EDIT THIS VALUE TO PREDICT YOUR TEXT'S LANGUAGE
my_vectorized_text = best_vectorizer.transform([input_text])
prediction = best_knn.predict(my_vectorized_text)[0]
print(f"Input: '{input_text}' -> Predicted: {prediction}")

Input: 'Chi va piano, va sano e va lontano' -> Predicted: Italian
